Use this notebook to define 3 categories of CNT density: 
Low, Mid, High

In [3]:
from pathlib import Path
from functools import reduce
from collections import defaultdict
import sys

import pandas as pd
import json
import matplotlib.pyplot as plt

import numpy as np

import scienceplots

# Resolve repo root robustly so paths are independent of notebook launch cwd.
repo_root = Path.cwd().resolve()
while not (repo_root / "src" / "cnt_project").exists():
    if repo_root.parent == repo_root:
        raise RuntimeError("Could not locate project root containing src/cnt_project")
    repo_root = repo_root.parent

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.append(str(src_root))

from cnt_project.io.paths import ProjectPaths

PATHS = ProjectPaths.from_here(src_root / "cnt_project" / "io" / "paths.py")
PATHS.ensure_outputs()

print(f"Project root: {PATHS.project_root}")
print(f"Data root: {PATHS.data_root}")

plt.style.use(['science', 'notebook', 'grid', 'no-latex'])

Project root: C:\Users\abd93000\PycharmProjects\cnt_project_v2
Data root: C:\Users\abd93000\PycharmProjects\cnt_project_v2\data


In [4]:
input_gt_json_path = (
    PATHS.data_root
    / "annotations_filtered_artifacts"
    / "test"
    / "COCO_mask"
    / "annotations.json"
)
with input_gt_json_path.open() as f:
    input_gt_json = json.load(f)

# Manually select density with thresholds of objects' number

In [5]:
image_id_to_filename = {img['id']: img['file_name'].split('.')[0] for img in input_gt_json['images']}

annotation_counts = {}
for ann in input_gt_json['annotations']:
    img_id = ann['image_id']
    annotation_counts[img_id] = annotation_counts.get(img_id, 0) + 1

# Collect all filenames with counts (including zero annotations)
filename_counts = []
all_filenames = set(image_id_to_filename.values())
annotated_filenames = set()
for img_id, count in annotation_counts.items():
    filename = image_id_to_filename.get(img_id)
    if filename:
        filename_counts.append((filename, count))
        annotated_filenames.add(filename)

# Add images with zero annotations
no_annotation_filenames = all_filenames - annotated_filenames
for fn in no_annotation_filenames:
    filename_counts.append((fn, 0))
    
def print_group(name, group):
    max_objects = max([c for fn_, c in filename_counts if fn_ in group], default=0)
    min_objects = min([c for fn_, c in filename_counts if fn_ in group], default=0)
    print(f"\nMax-min number of objects in {name} group: {min_objects}-{max_objects} \n")
    print(f"**{name}**")
    for fn in sorted(group):
        print(fn)

In [6]:
low_thresh = 30  # Threshold for low group
high_thresh = 100  # Threshold for high group

# Assign groups 
low_group = [fn for fn, c in filename_counts if c <= low_thresh]
mid_group = [fn for fn, c in filename_counts if low_thresh <= c <= high_thresh] 
high_group = [fn for fn, c in filename_counts if c > high_thresh]

# Print all groups
print_group("low", low_group)
print_group("mid", mid_group)
print_group("high", high_group)


Max-min number of objects in low group: 4-29 

**low**
400-1293-17-c10k1r5_pd_sp0_9_fl0_1_right
400-1293-17-c3k1r5_pd_sp0_3_fl0_1_left
400-1293-17-c3k1r5_pd_sp0_3_fl0_1_right
400-1293-17-c79k1r5_pd_sp0_6_fl0_1_right
400-1293-w14_c06k1r6_sp0_6_flow0_1_top_left
400-1605-W3_map_00034_top_left
400-1605-W8_map_00038_3

Max-min number of objects in mid group: 38-76 

**mid**
400-1293-23_3_0_20230307080848
400-1293-w14_c05k1r6_sp0_2_flow0_1_top_left
400-1293-w14_c09k1r6_sp0_1_flow0_1_bottom_right
400-1605-W1_map_00030_bottom_right
400-1605-W1_map_00039_bottom_left
400-1605-W2_map_00046_bottom_right
400-1605-W3_map_00034_1
400-1605-W3_map_00041_1
400-1605-W7_map_00021_4
400-1605-W7_map_00028_3
400-1605-W8_map_00006_bottom_right
400-1605-W8_map_00047_3
400-1605-W8_map_00049_1
400-1605-W8_map_00049_top_right

Max-min number of objects in high group: 104-255 

**high**
1293-22_3_2_20230306222551
400-1093-w11-c1p1-befo3-10sp-15flow_left
400-1093-w11-c1p2-befo3-20sp-15flow_left
400-1093-w13-c1-p1-

# Automatically select the density by grouping in 3 equal size groups

In [7]:
# Number of objects per image
image_object_counts = defaultdict(int)
for ann in input_gt_json['annotations']:
    image_object_counts[ann['image_id']] += 1

# Get a sorted list of (image_id, object_count) tuples by obect count
image_counts = list(image_object_counts.items())
image_counts.sort(key=lambda x: x[1])

# 3 equal-sized groups: low, mid, high
num_images = len(image_counts)
third = num_images // 3

low_density = image_counts[:third]
mid_density = image_counts[third:2*third]
high_density = image_counts[2*third:]

# Define thresholds based on group boundaries
low_max = low_density[-1][1] if low_density else 0
mid_max = mid_density[-1][1] if mid_density else 0

# Print results
print("Density thresholds (based on object count per image):")
print(f"Low density:     ≤ {low_max} objects")
print(f"Mid density: > {low_max} and ≤ {mid_max} objects")
print(f"High density:    > {mid_max} objects\n")

print("Number of images in each density group:")
print(f"Low:  {len(low_density)}")
print(f"Mid:  {len(mid_density)}")
print(f"High: {len(high_density)}")

Density thresholds (based on object count per image):
Low density:     ≤ 47 objects
Mid density: > 47 and ≤ 71 objects
High density:    > 71 objects

Number of images in each density group:
Low:  10
Mid:  10
High: 10
